# Custom NER model

In [1]:
import spacy
from spacy.tokens import DocBin
from spacy.training.example import Example

## Prepare training data

In [2]:
# spaCy expects data in a specific (text, annotations) format.
# Annotations are (start_char, end_char, entity_label) tuples.
TRAIN_DATA = [
    # Example 1: Highlighting a failure mode and a material spec
    ("The rotor showed high-cycle fatigue due to vibration, a common issue for titanium alloy 6-4.",
     {"entities": [
         (17, 34, "FAILURE_MODE"),       # high-cycle fatigue
         (68, 86, "MATERIAL_SPEC")       # titanium alloy 6-4
     ]}),
     
    # Example 2: Highlighting a sensor ID and a different failure mode
    ("Sensor S-431-A recorded an anomaly before catastrophic shear failure.",
     {"entities": [
         (7, 14, "SENSOR_ID"),          # S-431-A
         (41, 62, "FAILURE_MODE")        # catastrophic shear failure
     ]}),
     
    # Example 3: Different material and duration
    ("The composite structure, made of Carbon Fiber P75, maintained integrity for 500 hours.",
     {"entities": [
         (30, 48, "MATERIAL_SPEC"),       # Carbon Fiber P75
         (69, 78, "DURATION")            # 500 hours
     ]}),
     
    # Example 4: Multiple entities and different phrasing
    ("Test R-11 experienced thermal cracking failure after 120 cycles using Aluminum 7075.",
     {"entities": [
         (19, 36, "FAILURE_MODE"),       # thermal cracking failure
         (43, 53, "DURATION"),            # 120 cycles
         (60, 75, "MATERIAL_SPEC")       # Aluminum 7075
     ]}),
     
    # Example 5: Entity at the start/end of the sentence
    ("Inconel 718 was selected for its high resistance to creep failure.",
     {"entities": [
         (0, 12, "MATERIAL_SPEC"),       # Inconel 718
         (49, 62, "FAILURE_MODE")        # creep failure
     ]}),
    
    # Example 6: No specific domain entities (for robustness)
    ("The initial findings were reported to the engineering team on Tuesday.",
     {"entities": []})
]

## Create training pipeline

In [3]:
def create_doc_bin(data, vocab):
    db = DocBin()
    for text, annot in data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annot)
        db.add(example.reference)
    return db

In [4]:
# A new blank model for training
nlp = spacy.blank("en")
train_db = create_doc_bin(TRAIN_DATA, nlp.vocab)
train_db.to_disk("./train.spacy")

c:\Users\Shivam Sharma\AppData\Local\Programs\Python\Python311\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "The rotor showed high-cycle fatigue due to vibrati..." with entities "[(17, 34, 'FAILURE_MODE'), (68, 86, 'MATERIAL_SPEC...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\Shivam Sharma\AppData\Local\Programs\Python\Python311\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Sensor S-431-A recorded an anomaly before catastro..." with entities "[(7, 14, 'SENSOR_ID'), (41, 62, 'FAILURE_MODE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\Users\Shivam Sharma\AppData\Local\Progra

Run below command in bash for training

`python -m spacy train config.cfg     --output ./output_ner     --paths.train ./train.spacy     --paths.dev ./dev.spacy`

## Inference

In [5]:
output_dir = './output_ner/model-best'
nlp_ner = spacy.load(output_dir)
nlp_base = spacy.load("en_core_web_sm")

print(f"Successfully loaded model from: {output_dir}")

Successfully loaded model from: ./output_ner/model-best


In [6]:
# A sample text snippet that includes your custom entities
sample_text = (
    "The test was performed on sensor ID D-845, which confirmed a "
    "failure mode of 'Shear Stress Fracture' in the steel alloy AS-99-B, "
    "which requires further analysis."
    "Sensor S-431-A recorded an anomaly before catastrophic shear failure."
)

# Process the text
doc = nlp_base(sample_text)

# Print all detected entities
print("\n--- Detected Entities ---")
for ent in doc.ents:
    print(f"Label: {ent.label_:<20} Text: {ent.text}")

# --- Expected Output ---
# Label: SENSOR_ID            Text: D-845
# Label: MATERIAL_SPEC        Text: AS-99-B
# Label: FAILURE_MODE         Text: Shear Stress Fracture


--- Detected Entities ---
Label: WORK_OF_ART          Text: 'Shear Stress Fracture'
Label: PRODUCT              Text: AS-99-B
Label: PERSON               Text: Sensor S-431-A
